# Path B ORPO Colab Run (Extension Ready)

Run cells top-to-bottom in Colab extension.

Expected input file in session: `path_b_pairs.jsonl`.

In [1]:
!pip -q install --upgrade pip
# keep Colab base storage stack intact; only remove optional packages that broke this run path
!pip -q uninstall -y bitsandbytes triton
# install a tested ORPO stack
!pip -q install transformers==4.46.3 trl==0.11.4 peft==0.12.0 accelerate==0.34.2 datasets==3.2.0

print('Install complete. Runtime > Restart session is recommended once after this cell.')

In [2]:
import torch
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')
if not torch.cuda.is_available():
    raise RuntimeError(
        'GPU is OFF. In Colab: Runtime -> Change runtime type -> Hardware accelerator = GPU (T4), then rerun from top.'
    )

In [3]:
import os
from pathlib import Path

def find_training_file():
    candidates = [
        Path('/content/path_b_pairs.jsonl'),
        Path('/content/drive/MyDrive/path_b_pairs.jsonl'),
    ]
    for p in candidates:
        if p.exists():
            return p
    return None

p = find_training_file()
if p is not None:
    print('Found training pairs at:', p)
else:
    print('No training file found yet. Trying browser upload widget...')
    try:
        from google.colab import files
        uploaded = files.upload()
        if 'path_b_pairs.jsonl' in uploaded:
            print('Uploaded path_b_pairs.jsonl to /content')
        elif len(uploaded) > 0:
            first_name = list(uploaded.keys())[0]
            print('Uploaded', first_name, '- rename it to path_b_pairs.jsonl if needed.')
        else:
            print('No file selected in widget.')
    except Exception as e:
        print('Upload widget failed:', e)
        print('Use left sidebar Files -> Upload and select path_b_pairs.jsonl')

    p = find_training_file()
    if p is not None:
        print('Now found training pairs at:', p)
    else:
        print('Still not found. Optionally mount drive and place file at /content/drive/MyDrive/path_b_pairs.jsonl')

In [4]:
%%writefile train_path_b_colab.py
import json
import os
import random
from pathlib import Path

import numpy as np
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    default_data_collator,
)

SEED = 11
MAX_LEN = 768
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is False. Switch Colab runtime to GPU (T4).')

os.environ['BITSANDBYTES_NOWELCOME'] = '1'

model_id = 'Qwen/Qwen2.5-0.5B-Instruct'
revision = 'main'

candidate_paths = [
    Path('/content/path_b_pairs.jsonl'),
    Path('/content/drive/MyDrive/path_b_pairs.jsonl'),
]
train_file = None
for p in candidate_paths:
    if p.exists():
        train_file = str(p)
        break
if train_file is None:
    raise FileNotFoundError('Could not find path_b_pairs.jsonl in /content or /content/drive/MyDrive')

print('Using train file:', train_file)
print('GPU:', torch.cuda.get_device_name(0))

tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    revision=revision,
    torch_dtype=torch.float16,
)
model.config.use_cache = False
model.gradient_checkpointing_enable()

lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_cfg)

raw = load_dataset('json', data_files={'train': train_file})['train']
print('Train rows:', len(raw))

# Stable fallback: train on prompt+chosen only
def to_text(example):
    return {'text': example['prompt'] + '\n\n' + example['chosen']}

text_ds = raw.map(to_text, remove_columns=raw.column_names)

def tokenize_fn(batch):
    out = tokenizer(
        batch['text'],
        truncation=True,
        max_length=MAX_LEN,
        padding='max_length',
    )
    out['labels'] = [ids[:] for ids in out['input_ids']]
    return out

train_ds = text_ds.map(tokenize_fn, batched=True, remove_columns=text_ds.column_names)

args = TrainingArguments(
    output_dir='orpo_out',
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    max_steps=200,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    logging_steps=10,
    save_steps=100,
    eval_strategy='no',
    seed=SEED,
    report_to=[],
    fp16=True,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    data_collator=default_data_collator,
)

result = trainer.train()
Path('orpo_out').mkdir(exist_ok=True)
trainer.save_model('orpo_out/lora_adapter')
tokenizer.save_pretrained('orpo_out/lora_adapter')

metrics = {
    'train_loss': result.metrics.get('train_loss'),
    'train_runtime': result.metrics.get('train_runtime'),
    'train_steps_per_second': result.metrics.get('train_steps_per_second'),
    'seed': SEED,
    'model_id': model_id,
    'revision': revision,
    'train_file': train_file,
    'trainer': 'Transformers Trainer fallback (prompt+chosen SFT, fixed padding)',
}
print('metrics:', metrics)
with open('orpo_out/train_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

In [5]:
!python -u train_path_b_colab.py

In [6]:
import os
from google.colab import files

if os.path.isdir('orpo_out/lora_adapter'):
    !zip -r lora_adapter.zip orpo_out/lora_adapter
    files.download('lora_adapter.zip')
else:
    print('Adapter folder not found. Check training error output first.')